# Multi-Dataset Smoke Test for Turkish Legal RAG

This notebook tests whether the RAG pipeline can work with external Turkish legal datasets.

The goal is not full answer generation, but a retrieval smoke test:
- load external dataset files
- map dataset columns with configuration
- build FAISS and BM25 retrieval indexes
- run sample queries
- calculate basic retrieval metrics such as Hit@5 and Top-1 source match

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

data_path = f"{project_path}/data"
external_path = f"{data_path}/raw/external_datasets"
external_extracted_path = f"{external_path}/extracted"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("External datasets path:", external_path)
print("Extracted path:", external_extracted_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
External datasets path: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets
Extracted path: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [4]:
!pip install -q -U sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 48.4 MB/s eta 0:00:00


In [5]:
import os
import re
import glob
import json
import zipfile

import numpy as np
import pandas as pd
import faiss

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [6]:
zip_files = glob.glob(f"{external_path}/*.zip")

print("Zip count:", len(zip_files))
for zip_file in zip_files:
    print(zip_file)

Zip count: 2
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/dataset_and_goldTest.zip
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/CENG493_Starlar.zip


In [7]:
os.makedirs(external_extracted_path, exist_ok=True)

for zip_path in zip_files:
    zip_name = os.path.splitext(os.path.basename(zip_path))[0]
    target_dir = os.path.join(external_extracted_path, zip_name)

    os.makedirs(target_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir)

    print("Extracted:", zip_path, "->", target_dir)

Extracted: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/dataset_and_goldTest.zip -> /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest
Extracted: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/CENG493_Starlar.zip -> /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar


In [8]:
for root, dirs, files in os.walk(external_extracted_path):
    level = root.replace(external_extracted_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

extracted/
  dataset_and_goldTest/
    dataset_and_goldTest/
      qa_benchmark_gold.csv
      README.md
      turk_rag_corpus.csv
  CENG493_Starlar/
    corpus.jsonl
    embedding.jsonl
    gold_benchmark.json
    llm.jsonl
    rag_eval.json
    reranker.jsonl


In [9]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]


def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def build_faiss_bm25_index(df, text_col, embedding_model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    embedding_model = SentenceTransformer(embedding_model_name)

    texts = df[text_col].astype(str).tolist()

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype("float32")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    tokenized_corpus = [
        simple_turkish_tokenize(text)
        for text in texts
    ]

    bm25 = BM25Okapi(tokenized_corpus)

    return embedding_model, index, bm25


def hybrid_retrieve(query, df, text_col, id_col, source_col, embedding_model, index, bm25, k=5, alpha=0.5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(df))
    ])

    bm25_scores = np.array(
        bm25.get_scores(simple_turkish_tokenize(query))
    )

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": df.iloc[idx][id_col],
            "source": df.iloc[idx][source_col],
            "score": float(final_scores[idx]),
            "text": df.iloc[idx][text_col]
        })

    return results

Part A

In [10]:
turk_corpus_candidates = glob.glob(
    f"{external_extracted_path}/**/turk_rag_corpus.csv",
    recursive=True
)

turk_gold_candidates = glob.glob(
    f"{external_extracted_path}/**/qa_benchmark_gold.csv",
    recursive=True
)

print("Corpus candidates:")
for p in turk_corpus_candidates:
    print(p)

print("\nGold QA candidates:")
for p in turk_gold_candidates:
    print(p)

turk_corpus_path = turk_corpus_candidates[0]
turk_gold_path = turk_gold_candidates[0]

print("\nUsing corpus:", turk_corpus_path)
print("Using gold QA:", turk_gold_path)

Corpus candidates:
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/turk_rag_corpus.csv

Gold QA candidates:
/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/qa_benchmark_gold.csv

Using corpus: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/turk_rag_corpus.csv
Using gold QA: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/qa_benchmark_gold.csv


In [11]:
TURK_RAG_CONFIG = {
    "dataset_name": "external_turk_rag_25law",

    "corpus_path": turk_corpus_path,
    "qa_path": turk_gold_path,

    "text_column": "retrieval_text",
    "context_column": "context",
    "source_column": "kaynak",
    "chunk_id_column": "context_key",

    "question_column": "soru",
    "answer_column": "cevap",
    "gold_context_column": "context",
    "gold_source_column": "kaynak",
    "gold_chunk_id_column": "context_key"
}

TURK_RAG_CONFIG

{'dataset_name': 'external_turk_rag_25law',
 'corpus_path': '/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/turk_rag_corpus.csv',
 'qa_path': '/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/dataset_and_goldTest/dataset_and_goldTest/qa_benchmark_gold.csv',
 'text_column': 'retrieval_text',
 'context_column': 'context',
 'source_column': 'kaynak',
 'chunk_id_column': 'context_key',
 'question_column': 'soru',
 'answer_column': 'cevap',
 'gold_context_column': 'context',
 'gold_source_column': 'kaynak',
 'gold_chunk_id_column': 'context_key'}

In [12]:
turk_chunks_df = pd.read_csv(TURK_RAG_CONFIG["corpus_path"])
turk_qa_df = pd.read_csv(TURK_RAG_CONFIG["qa_path"])

print("Turk RAG corpus:", turk_chunks_df.shape)
print("Turk RAG QA:", turk_qa_df.shape)

print("\nCorpus columns:")
print(turk_chunks_df.columns.tolist())

print("\nQA columns:")
print(turk_qa_df.columns.tolist())

display(turk_chunks_df.head())
display(turk_qa_df.head())

Turk RAG corpus: (6326, 8)
Turk RAG QA: (290, 29)

Corpus columns:
['kaynak', 'madde_no', 'context_key', 'context', 'retrieval_text', 'chunk_strategy', 'kanun_no', 'url']

QA columns:
['row_id', 'soru', 'cevap', 'context', 'kaynak', 'Score', 'source_origin', 'question_template', 'selection_score', 'answer_in_chunk', 'chunk_strategy', 'chunk_len', 'context_key', 'question_key', 'kanun_adi', 'madde_nolari_soru', 'madde_nolari_context', 'madde_no', 'bolum_fasil', 'heading_count', 'answer_support_overlap', 'question_context_overlap', 'question_answer_overlap', 'answer_in_context_substring', 'open_ended_question', 'answer_mentions_other_source', 'context_starts_mid_sentence', 'score_valid', 'manual_review_reason']


,kaynak,madde_no,context_key,context,retrieval_text,chunk_strategy,kanun_no,url
0,Türkiye Cumhuriyeti Anayasası,1,Türkiye Cumhuriyeti Anayasası|madde-1|chunk-0,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyeti Anayasası\nTürkiye Devleti...,statute_full,2709,https://www.mevzuat.gov.tr/MevzuatMetin/1.5.27...
1,Türkiye Cumhuriyeti Anayasası,2,Türkiye Cumhuriyeti Anayasası|madde-2|chunk-0,II. Cumhuriyetin nitelikleri Türkiye Cumhuriye...,Türkiye Cumhuriyeti Anayasası\nII. Cumhuriyeti...,statute_full,2709,https://www.mevzuat.gov.tr/MevzuatMetin/1.5.27...
2,Türkiye Cumhuriyeti Anayasası,3,Türkiye Cumhuriyeti Anayasası|madde-3|chunk-0,"III. Devletin bütünlüğü, Resmî dili, bayrağı, ...",Türkiye Cumhuriyeti Anayasası\nIII. Devletin b...,statute_full,2709,https://www.mevzuat.gov.tr/MevzuatMetin/1.5.27...
3,Türkiye Cumhuriyeti Anayasası,4,Türkiye Cumhuriyeti Anayasası|madde-4|chunk-0,IV. Değiştirilemeyecek hükümler Anayasanın 1 i...,Türkiye Cumhuriyeti Anayasası\nIV. Değiştirile...,statute_full,2709,https://www.mevzuat.gov.tr/MevzuatMetin/1.5.27...
4,Türkiye Cumhuriyeti Anayasası,5,Türkiye Cumhuriyeti Anayasası|madde-5|chunk-0,2 V. Devletin temel amaç ve görevleri Devletin...,Türkiye Cumhuriyeti Anayasası\n2 V. Devletin t...,statute_full,2709,https://www.mevzuat.gov.tr/MevzuatMetin/1.5.27...


,row_id,soru,cevap,context,kaynak,Score,source_origin,question_template,selection_score,answer_in_chunk,...,heading_count,answer_support_overlap,question_context_overlap,question_answer_overlap,answer_in_context_substring,open_ended_question,answer_mentions_other_source,context_starts_mid_sentence,score_valid,manual_review_reason
0,1514.0,"'Belge' tanımı, Bilgi Edinme Hakkı Kanunu'nda ...","Bilgi Edinme Hakkı Kanunu'nda 'belge', kurum v...",Madde 3- Bu Kanunda geçen; a) Kurum ve kuruluş...,Bilgi Edinme Kanunu,9,kaggle_batuhankalem,"'belge' tanımı, bilgi edinme hakkı kanunu'nda ...",105.0,False,...,0.0,0.723404,0.375000,0.875000,False,False,False,False,True,NaN
1,1623.0,Yayımlanmış veya kamuya açıklanmış bilgiler bi...,"Kurum ve kuruluşlarca yayımlanmış veya yayın, ...",Madde 8- Kurum ve kuruluşlarca yayımlanmış vey...,Bilgi Edinme Kanunu,9,kaggle_batuhankalem,yayımlanmış veya kamuya açıklanmış bilgiler bi...,79.0,False,...,0.0,0.800000,0.777778,0.777778,False,False,False,False,True,NaN
2,1628.0,Bilgi veya belgeye erişim süresi ne kadardır?,"Kurum ve kuruluşlar, başvuru üzerine istenen b...","Madde 11- Kurum ve kuruluşlar, başvuru üzerine...",Bilgi Edinme Kanunu,9,kaggle_batuhankalem,bilgi veya belgeye erişim süresi ne kadardır?,108.0,False,...,0.0,0.813953,0.600000,0.600000,False,False,False,False,True,NaN
3,1642.0,Bilgi edinme başvurusu hangi iletişim araçları...,"Bilgi edinme başvurusu, kişinin kimliğinin ve ...",ÜÇÜNCÜ BÖLÜM Bilgi Edinme Başvurusu Başvuru us...,Bilgi Edinme Kanunu,9,kaggle_batuhankalem,bilgi edinme başvurusu hangi iletişim araçları...,84.0,False,...,0.0,0.866667,1.000000,1.000000,False,False,False,False,True,NaN
4,1656.0,İdari soruşturmaya ilişkin hangi bilgilerin ka...,"İdari soruşturmaya ilişkin, kişilerin özel hay...",Madde 19- Kurum ve kuruluşların yetkili biriml...,Bilgi Edinme Kanunu,9,kaggle_batuhankalem,i̇dari soruşturmaya ilişkin hangi bilgilerin k...,102.0,False,...,0.0,0.868421,0.428571,1.000000,False,False,False,False,True,NaN


In [13]:
print("Corpus source counts:")
display(turk_chunks_df[TURK_RAG_CONFIG["source_column"]].value_counts().head(30))

print("QA source counts:")
display(turk_qa_df[TURK_RAG_CONFIG["gold_source_column"]].value_counts().head(30))

Corpus source counts:


,count
kaynak,
Türk Ticaret Kanunu,1540
Türk Medeni Kanunu,1023
Türk Borçlar Kanunu,648
Hukuk Muhakemeleri Kanunu,458
Ceza Muhakemesi Kanunu,359
Türk Ceza Kanunu,352
Devlet Memurları Kanunu,333
Sosyal Sigortalar ve Genel Sağlık Sigortası Kanunu,261
Avukatlık Kanunu,212


QA source counts:


,count
kaynak,
Türk Medeni Kanunu,50
Türk Ceza Kanunu,40
Türk Borçlar Kanunu,36
Ceza Muhakemesi Kanunu,24
Türkiye Cumhuriyeti Anayasası,22
Türkiye Cumhuriyeti İş Kanunu,13
Türk Bayrağı Tüzüğü,8
Bilgi Edinme Kanunu,7
Avukatlık Kanunu,5


In [14]:
TEXT_COL = TURK_RAG_CONFIG["text_column"]
SOURCE_COL = TURK_RAG_CONFIG["source_column"]
CHUNK_ID_COL = TURK_RAG_CONFIG["chunk_id_column"]

QUESTION_COL = TURK_RAG_CONFIG["question_column"]
ANSWER_COL = TURK_RAG_CONFIG["answer_column"]

turk_retrieval_df = turk_chunks_df.copy()

turk_retrieval_df = turk_retrieval_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)

turk_retrieval_df[CHUNK_ID_COL] = turk_retrieval_df[CHUNK_ID_COL].astype(str)
turk_retrieval_df[SOURCE_COL] = turk_retrieval_df[SOURCE_COL].astype(str)

print(turk_retrieval_df.shape)
display(turk_retrieval_df[[CHUNK_ID_COL, SOURCE_COL, TEXT_COL]].head())

(6326, 8)


,context_key,kaynak,retrieval_text
0,Türkiye Cumhuriyeti Anayasası|madde-1|chunk-0,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası\nTürkiye Devleti...
1,Türkiye Cumhuriyeti Anayasası|madde-2|chunk-0,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası\nII. Cumhuriyeti...
2,Türkiye Cumhuriyeti Anayasası|madde-3|chunk-0,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası\nIII. Devletin b...
3,Türkiye Cumhuriyeti Anayasası|madde-4|chunk-0,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası\nIV. Değiştirile...
4,Türkiye Cumhuriyeti Anayasası|madde-5|chunk-0,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası\n2 V. Devletin t...


In [15]:
turk_embedding_model, turk_index, turk_bm25 = build_faiss_bm25_index(
    df=turk_retrieval_df,
    text_col=TEXT_COL
)

print("FAISS vectors:", turk_index.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

FAISS vectors: 6326


In [16]:
turk_sample_qa_df = turk_qa_df.sample(n=5, random_state=42).reset_index(drop=True)

for i, row in turk_sample_qa_df.iterrows():
    question = row[QUESTION_COL]
    expected_answer = row[ANSWER_COL]
    gold_source = row[TURK_RAG_CONFIG["gold_source_column"]]
    gold_chunk_id = str(row[TURK_RAG_CONFIG["gold_chunk_id_column"]])

    results = hybrid_retrieve(
        query=question,
        df=turk_retrieval_df,
        text_col=TEXT_COL,
        id_col=CHUNK_ID_COL,
        source_col=SOURCE_COL,
        embedding_model=turk_embedding_model,
        index=turk_index,
        bm25=turk_bm25,
        k=5,
        alpha=0.5
    )

    retrieved_ids = [str(r["chunk_id"]) for r in results]

    print("=" * 120)
    print("INDEX:", i)
    print("QUESTION:", question)
    print("EXPECTED:", str(expected_answer)[:300])
    print("GOLD SOURCE:", gold_source)
    print("GOLD CHUNK ID:", gold_chunk_id)
    print("HIT@5:", gold_chunk_id in retrieved_ids)

    print("\nTOP RESULTS:")
    for r in results:
        print("-" * 80)
        print("Rank:", r["rank"])
        print("Chunk ID:", r["chunk_id"])
        print("Source:", r["source"])
        print("Score:", r["score"])
        print(str(r["text"])[:500])

INDEX: 0
QUESTION: Rızaya dayalı olsa bile, yetkisiz bir kişi tarafından yapılan kısırlaştırma eylemi ne kadar hapis cezası gerektirir?
EXPECTED: Türk Ceza Kanunu'nun 101. maddesinin 2. fıkrasına göre, rızaya dayalı olsa bile, kısırlaştırma fiilinin yetkili olmayan bir kişi tarafından işlenmesi hâlinde, bir yıldan üç yıla kadar hapis cezasına hükmolunur.
GOLD SOURCE: Türk Ceza Kanunu
GOLD CHUNK ID: madde 100. - (1) gebelik süresi on haftadan fazla olan kadının çocuğunu isteyerek düşürmesi hâlinde, bir yıla kadar hapis veya adlî para cezasına hükmolunur. kısırlaştırma madde 101. - (1) bir erkek veya kadını rızası olmaksızın kısırlaştıran kimse, üç yıldan altı yıla kadar hapis cezası ile cezalandırılır. fiil, kısırlaştırma işlemi yapma yetkisi olmayan bir kimse tarafından yapılırsa, ceza üçte bir oranında artırılır. (2) rızaya dayalı olsa bile, kısırlaştırma fiilinin yetkili olmayan bir kişi tarafından işlenmesi hâlinde, bir yıldan üç yıla kadar hapis cezasına hükmolunur.
HIT@5: False

T

In [17]:
turk_eval_df = turk_qa_df.sample(n=min(50, len(turk_qa_df)), random_state=42).reset_index(drop=True)

turk_retrieval_results = []

for i, row in tqdm(turk_eval_df.iterrows(), total=len(turk_eval_df)):
    question = row[QUESTION_COL]
    gold_source = str(row[TURK_RAG_CONFIG["gold_source_column"]])
    gold_chunk_id = str(row[TURK_RAG_CONFIG["gold_chunk_id_column"]])

    results = hybrid_retrieve(
        query=question,
        df=turk_retrieval_df,
        text_col=TEXT_COL,
        id_col=CHUNK_ID_COL,
        source_col=SOURCE_COL,
        embedding_model=turk_embedding_model,
        index=turk_index,
        bm25=turk_bm25,
        k=5,
        alpha=0.5
    )

    retrieved_ids = [str(r["chunk_id"]) for r in results]
    retrieved_sources = [str(r["source"]) for r in results]

    turk_retrieval_results.append({
        "dataset": TURK_RAG_CONFIG["dataset_name"],
        "index": i,
        "question": question,
        "gold_source": gold_source,
        "gold_chunk_id": gold_chunk_id,
        "top1_chunk_id": retrieved_ids[0],
        "top1_source": retrieved_sources[0],
        "hit_at_1": gold_chunk_id == retrieved_ids[0],
        "hit_at_5": gold_chunk_id in retrieved_ids,
        "top1_source_match": gold_source == retrieved_sources[0],
        "top5_source_match": gold_source in retrieved_sources
    })

turk_retrieval_results_df = pd.DataFrame(turk_retrieval_results)

turk_retrieval_results_df.head()

100%|██████████| 50/50 [00:04<00:00, 10.96it/s]


,dataset,index,question,gold_source,gold_chunk_id,top1_chunk_id,top1_source,hit_at_1,hit_at_5,top1_source_match,top5_source_match
0,external_turk_rag_25law,0,"Rızaya dayalı olsa bile, yetkisiz bir kişi tar...",Türk Ceza Kanunu,madde 100. - (1) gebelik süresi on haftadan fa...,Türk Ceza Kanunu|madde-101|chunk-0,Türk Ceza Kanunu,False,False,True,True
1,external_turk_rag_25law,1,Hangi fiillerin kabahat uyarınca öngörülen huk...,Kabahatler Kanunu,Kabahatler Kanunu|madde-4|chunk-0,Kabahatler Kanunu|madde-4|chunk-0,Kabahatler Kanunu,True,True,True,True
2,external_turk_rag_25law,2,"Devralan, garanti ile yükümlü olan devredenden...",Türk Borçlar Kanunu,"madde 191- alacak, bir edim karşılığında devre...",Türk Borçlar Kanunu|madde-193|chunk-0,Türk Borçlar Kanunu,False,False,True,True
3,external_turk_rag_25law,3,Yürütme yetkisi kim tarafından kullanılır?,Türkiye Cumhuriyeti Anayasası,"madde 5 – devletin temel amaç ve görevleri, tü...",Türkiye Cumhuriyeti Anayasası|madde-8|chunk-0,Türkiye Cumhuriyeti Anayasası,False,False,True,True
4,external_turk_rag_25law,4,"Eşlerden birinin ölümü hâlinde, sağ kalan eş, ...",Türk Medeni Kanunu,madde 651- değerinde önemli azalma olmadan böl...,Türk Medeni Kanunu|madde-652|chunk-0,Türk Medeni Kanunu,False,False,True,True


In [18]:
turk_summary = {
    "dataset": TURK_RAG_CONFIG["dataset_name"],
    "corpus_rows": len(turk_retrieval_df),
    "qa_rows": len(turk_qa_df),
    "eval_sample_size": len(turk_retrieval_results_df),
    "hit_at_1": turk_retrieval_results_df["hit_at_1"].mean(),
    "hit_at_5": turk_retrieval_results_df["hit_at_5"].mean(),
    "top1_source_match": turk_retrieval_results_df["top1_source_match"].mean(),
    "top5_source_match": turk_retrieval_results_df["top5_source_match"].mean()
}

turk_summary_df = pd.DataFrame([turk_summary])
turk_summary_df

,dataset,corpus_rows,qa_rows,eval_sample_size,hit_at_1,hit_at_5,top1_source_match,top5_source_match
0,external_turk_rag_25law,6326,290,50,0.12,0.2,0.8,0.9


In [19]:
turk_retrieval_results_df.to_csv(
    f"{metrics_path}/external_turk_rag_25law_smoke_retrieval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

turk_summary_df.to_csv(
    f"{metrics_path}/external_turk_rag_25law_smoke_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Turk RAG smoke test results saved.")

Turk RAG smoke test results saved.


Part B

In [20]:
starlar_corpus_candidates = glob.glob(
    f"{external_extracted_path}/**/corpus.jsonl",
    recursive=True
)

starlar_rag_eval_candidates = glob.glob(
    f"{external_extracted_path}/**/rag_eval.json",
    recursive=True
)

starlar_gold_candidates = glob.glob(
    f"{external_extracted_path}/**/gold_benchmark.json",
    recursive=True
)

print("Starlar corpus candidates:", starlar_corpus_candidates)
print("Starlar rag_eval candidates:", starlar_rag_eval_candidates)
print("Starlar gold candidates:", starlar_gold_candidates)

starlar_corpus_path = starlar_corpus_candidates[0]
starlar_rag_eval_path = starlar_rag_eval_candidates[0]
starlar_gold_path = starlar_gold_candidates[0]

print("\nUsing corpus:", starlar_corpus_path)
print("Using rag_eval:", starlar_rag_eval_path)
print("Using gold benchmark:", starlar_gold_path)

Starlar corpus candidates: ['/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl']
Starlar rag_eval candidates: ['/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json']
Starlar gold candidates: ['/content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/gold_benchmark.json']

Using corpus: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl
Using rag_eval: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json
Using gold benchmark: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/gold_benchmark.json


In [21]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [22]:
starlar_corpus_records = load_jsonl(starlar_corpus_path)
starlar_rag_eval_records = load_json(starlar_rag_eval_path)
starlar_gold_records = load_json(starlar_gold_path)

print("Starlar corpus records:", len(starlar_corpus_records))
print("Starlar rag_eval records:", len(starlar_rag_eval_records))
print("Starlar gold benchmark records:", len(starlar_gold_records))

print("\nSample corpus record:")
print(starlar_corpus_records[0])

print("\nSample rag_eval record:")
print(starlar_rag_eval_records[0])

Starlar corpus records: 7579
Starlar rag_eval records: 1000
Starlar gold benchmark records: 240

Sample corpus record:
{'id': 'oricon_anayasa_000001', 'text': 'Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.', 'title': 'Anayasa Hukuku ve Temel Haklar', 'metadata': {'source': 'ORICON', 'source_file': 'ORICON_clean_legal_source_for_chunking.txt', 'doc_id': 'oricon_legal_source', 'chunk_id': 'oricon_anayasa_000001', 'chunk_index': 1, 'category_chunk_index': 1, 'category': 'Anayasa Hukuku ve Temel Haklar', 'category_slug': 'anayasa', 'semantic_topic': 'susma hakkı', 'legal_concepts': ['susma hakkı'], 'citation_label': 'ORICON - Anayasa Hukuku ve Temel Haklar - oricon_anayasa_000001', 'verification_required': False, 'original_flags': None, 'quality_flags': None, 'source_location': {'line_start': 11, 'line_end': 11, 'char_start': 370, 'char_end': 477}, 'source_record_count': 1, 'chunk_token_count_approx': 16, 'chunk_char_count': 107

In [23]:
starlar_rows = []

for record in starlar_corpus_records:
    metadata = record.get("metadata", {}) or {}

    starlar_rows.append({
        "chunk_id": record.get("id"),
        "text": record.get("text"),
        "title": record.get("title"),
        "source": metadata.get("source"),
        "category": metadata.get("category"),
        "citation_label": metadata.get("citation_label")
    })

starlar_df = pd.DataFrame(starlar_rows)

starlar_df = starlar_df.dropna(subset=["text", "chunk_id"]).reset_index(drop=True)

print(starlar_df.shape)
display(starlar_df.head())

print("Source counts:")
display(starlar_df["source"].value_counts().head(20))

(7579, 6)


,chunk_id,text,title,source,category,citation_label
0,oricon_anayasa_000001,"Susma hakkı, kişinin kendi lehine veya aleyhin...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
1,oricon_anayasa_000003,"Düşünce özgürlüğü, düşünce ve kanaatlerin çeşi...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
2,oricon_anayasa_000004,"İfade özgürlüğü, demokratik toplumların temeli...",Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
3,oricon_anayasa_000005,Anayasada düzenlenen temel hak ve özgürlüklerd...,Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...
4,oricon_anayasa_000006,Demokratik bir toplumda eleştiri hakkının koru...,Anayasa Hukuku ve Temel Haklar,ORICON,Anayasa Hukuku ve Temel Haklar,ORICON - Anayasa Hukuku ve Temel Haklar - oric...


Source counts:


,count
source,
ORICON,3742
TURKISH_LAW_ESKI_LOW_RISK_ONLY,1727
YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,1242
TRAIN_LOW_RISK_ONLY,538
TURKISH_LAWCHATBOT_LOW_RISK_ONLY,330


In [24]:
starlar_embedding_model, starlar_index, starlar_bm25 = build_faiss_bm25_index(
    df=starlar_df,
    text_col="text"
)

print("Starlar FAISS vectors:", starlar_index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/237 [00:00<?, ?it/s]

Starlar FAISS vectors: 7579


In [25]:
starlar_sample_eval = pd.DataFrame(starlar_rag_eval_records).sample(n=5, random_state=42).reset_index(drop=True)

for i, row in starlar_sample_eval.iterrows():
    query = row["query"]
    gold_chunk_ids = [str(x) for x in row["gold_chunk_ids"]]

    results = hybrid_retrieve(
        query=query,
        df=starlar_df,
        text_col="text",
        id_col="chunk_id",
        source_col="source",
        embedding_model=starlar_embedding_model,
        index=starlar_index,
        bm25=starlar_bm25,
        k=5,
        alpha=0.5
    )

    retrieved_ids = [str(r["chunk_id"]) for r in results]

    print("=" * 120)
    print("INDEX:", i)
    print("QUERY:", query)
    print("GOLD CHUNK IDS:", gold_chunk_ids)
    print("HIT@5:", any(gold_id in retrieved_ids for gold_id in gold_chunk_ids))

    print("\nTOP RESULTS:")
    for r in results:
        print("-" * 80)
        print("Rank:", r["rank"])
        print("Chunk ID:", r["chunk_id"])
        print("Source:", r["source"])
        print("Score:", r["score"])
        print(str(r["text"])[:500])

INDEX: 0
QUERY: Türk Medenî Kanunu madde 76 kapsamında BİRİNCİ KİTAP konusunda Madde Bütün üyelerin araya gelmeksizin yazılı katılımıyla alınan düzenlemesi nasıl açıklanır?
GOLD CHUNK IDS: ['turkish_law_eski_4721_turk_medeni_kanunu_m76']
HIT@5: True

TOP RESULTS:
--------------------------------------------------------------------------------
Rank: 1
Chunk ID: turkish_law_eski_4721_turk_medeni_kanunu_m76
Source: TURKISH_LAW_ESKI_LOW_RISK_ONLY
Score: 0.9452149271965027
Madde 76- Bütün üyelerin bir araya gelmeksizin yazılı katılımıyla alınan kararlar ile.
dernek üyelerinin tamamının kanunda yazılı çağrı usulüne uymaksızın bir araya gelerek aldığı.
kararlar geçerlidir.
Bu şekilde karar alınması olağan toplantı yerine geçmez.
--------------------------------------------------------------------------------
Rank: 2
Chunk ID: turkish_law_eski_5271_ceza_muhakemesi_kanunu_m243
Source: TURKISH_LAW_ESKI_LOW_RISK_ONLY
Score: 0.6747980713844299
Madde 243 – (1) Katılan, vazgeçerse veya ölürse katılm

In [26]:
starlar_eval_df = pd.DataFrame(starlar_rag_eval_records).sample(n=min(100, len(starlar_rag_eval_records)), random_state=42).reset_index(drop=True)

starlar_retrieval_results = []

for i, row in tqdm(starlar_eval_df.iterrows(), total=len(starlar_eval_df)):
    query = row["query"]
    gold_chunk_ids = [str(x) for x in row["gold_chunk_ids"]]
    gold_source = str(row.get("source", ""))

    results = hybrid_retrieve(
        query=query,
        df=starlar_df,
        text_col="text",
        id_col="chunk_id",
        source_col="source",
        embedding_model=starlar_embedding_model,
        index=starlar_index,
        bm25=starlar_bm25,
        k=5,
        alpha=0.5
    )

    retrieved_ids = [str(r["chunk_id"]) for r in results]
    retrieved_sources = [str(r["source"]) for r in results]

    hit_at_1 = any(gold_id == retrieved_ids[0] for gold_id in gold_chunk_ids)
    hit_at_5 = any(gold_id in retrieved_ids for gold_id in gold_chunk_ids)

    starlar_retrieval_results.append({
        "dataset": "ceng493_starlar",
        "index": i,
        "query": query,
        "gold_chunk_ids": "; ".join(gold_chunk_ids),
        "gold_source": gold_source,
        "top1_chunk_id": retrieved_ids[0],
        "top1_source": retrieved_sources[0],
        "hit_at_1": hit_at_1,
        "hit_at_5": hit_at_5,
        "top1_source_match": gold_source == retrieved_sources[0],
        "top5_source_match": gold_source in retrieved_sources
    })

starlar_retrieval_results_df = pd.DataFrame(starlar_retrieval_results)

starlar_retrieval_results_df.head()

100%|██████████| 100/100 [00:16<00:00,  5.98it/s]


,dataset,index,query,gold_chunk_ids,gold_source,top1_chunk_id,top1_source,hit_at_1,hit_at_5,top1_source_match,top5_source_match
0,ceng493_starlar,0,Türk Medenî Kanunu madde 76 kapsamında BİRİNCİ...,turkish_law_eski_4721_turk_medeni_kanunu_m76,TURKISH_LAW_ESKI_LOW_RISK_ONLY,turkish_law_eski_4721_turk_medeni_kanunu_m76,TURKISH_LAW_ESKI_LOW_RISK_ONLY,True,True,True,True
1,ceng493_starlar,1,"Hukuk Genel Kurulu 2015/231 E., 2015/1467 K. s...",yargitay_1273_aile_hukuku_004,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,yargitay_1273_aile_hukuku_001,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,False,False,True,True
2,ceng493_starlar,2,"Hukuk Genel Kurulu 2013/1910 E., 2015/1203 K. ...",yargitay_0949_icra_ve_iflas_hukuku_004,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,yargitay_0949_icra_ve_iflas_hukuku_001,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,False,False,True,True
3,ceng493_starlar,3,Türk Medenî Kanunu madde 879 kapsamında DÖRDÜN...,turkish_law_eski_4721_turk_medeni_kanunu_m879,TURKISH_LAW_ESKI_LOW_RISK_ONLY,turkish_law_eski_4721_turk_medeni_kanunu_m879,TURKISH_LAW_ESKI_LOW_RISK_ONLY,True,True,True,True
4,ceng493_starlar,4,Medeni Hukuk - Miras/Aile/Eşya/Kişiler alanınd...,oricon_medeni_000411,ORICON,oricon_medeni_000411,ORICON,True,True,True,True


In [27]:
starlar_summary = {
    "dataset": "ceng493_starlar",
    "corpus_rows": len(starlar_df),
    "rag_eval_rows": len(starlar_rag_eval_records),
    "eval_sample_size": len(starlar_retrieval_results_df),
    "hit_at_1": starlar_retrieval_results_df["hit_at_1"].mean(),
    "hit_at_5": starlar_retrieval_results_df["hit_at_5"].mean(),
    "top1_source_match": starlar_retrieval_results_df["top1_source_match"].mean(),
    "top5_source_match": starlar_retrieval_results_df["top5_source_match"].mean()
}

starlar_summary_df = pd.DataFrame([starlar_summary])
starlar_summary_df

,dataset,corpus_rows,rag_eval_rows,eval_sample_size,hit_at_1,hit_at_5,top1_source_match,top5_source_match
0,ceng493_starlar,7579,1000,100,0.77,0.89,1.0,1.0


In [28]:
starlar_retrieval_results_df.to_csv(
    f"{metrics_path}/external_starlar_smoke_retrieval_results.csv",
    index=False,
    encoding="utf-8-sig"
)

starlar_summary_df.to_csv(
    f"{metrics_path}/external_starlar_smoke_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Starlar smoke test results saved.")

Starlar smoke test results saved.


PART C — Combined summary

In [29]:
multi_dataset_summary_df = pd.concat(
    [turk_summary_df, starlar_summary_df],
    ignore_index=True
)

multi_dataset_summary_df

,dataset,corpus_rows,qa_rows,eval_sample_size,hit_at_1,hit_at_5,top1_source_match,top5_source_match,rag_eval_rows
0,external_turk_rag_25law,6326,290.0,50,0.12,0.20,0.8,0.9,NaN
1,ceng493_starlar,7579,NaN,100,0.77,0.89,1.0,1.0,1000.0


In [30]:
multi_dataset_summary_df.to_csv(
    f"{metrics_path}/multi_dataset_smoke_test_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Multi-dataset smoke test summary saved.")

Multi-dataset smoke test summary saved.


In [31]:
clean_multi_dataset_summary_df = multi_dataset_summary_df.copy()

clean_multi_dataset_summary_df["benchmark_rows"] = clean_multi_dataset_summary_df["qa_rows"].fillna(
    clean_multi_dataset_summary_df["rag_eval_rows"]
)

clean_multi_dataset_summary_df = clean_multi_dataset_summary_df[
    [
        "dataset",
        "corpus_rows",
        "benchmark_rows",
        "eval_sample_size",
        "hit_at_1",
        "hit_at_5",
        "top1_source_match",
        "top5_source_match"
    ]
]

clean_multi_dataset_summary_df

,dataset,corpus_rows,benchmark_rows,eval_sample_size,hit_at_1,hit_at_5,top1_source_match,top5_source_match
0,external_turk_rag_25law,6326,290.0,50,0.12,0.20,0.8,0.9
1,ceng493_starlar,7579,1000.0,100,0.77,0.89,1.0,1.0


In [32]:
clean_multi_dataset_summary_df.to_csv(
    f"{metrics_path}/multi_dataset_smoke_test_clean_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Clean multi-dataset smoke test summary saved.")

Clean multi-dataset smoke test summary saved.


## Multi-Dataset Smoke Test Conclusion

The multi-dataset smoke test showed that the retrieval pipeline can be adapted to different Turkish legal datasets using configurable corpus and evaluation mappings.

For the external 25-law Turkish RAG dataset, the system achieved a Top-1 source match of 0.80 and a Top-5 source match of 0.90. However, exact chunk-level Hit@5 was 0.20, indicating that the system often retrieves from the correct legal source but does not always retrieve the exact gold chunk.

For the CENG493 Starlar dataset, retrieval performance was much stronger. The system achieved Hit@1 of 0.77 and Hit@5 of 0.89, with perfect source-level matching. This indicates that the retrieval pipeline works well when the corpus and evaluation chunk IDs are well aligned.

Overall, the smoke test confirms that the core retrieval pipeline can run on external datasets without changing the main retrieval logic. Dataset-specific column mappings and corpus/evaluation alignment are important for reliable performance.